# Lab 4 - Prepare Silver Updates

Cleans, validates, deduplicates, and materializes a durable staging table. This notebook has no final Silver side effects, so SCD Type 1 and Type 2 can safely reuse the same prepared input.


In [0]:
%run ./lab4_00_config


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


In [0]:
bronze_df = spark.table(bronze_table)

def clean_string(col_name):
    return F.when(F.length(F.trim(F.col(col_name))) == 0, None).otherwise(F.trim(F.col(col_name)))

known_columns = [
    "show_id", "type", "title", "director", "cast", "country", "date_added",
    "release_year", "rating", "duration", "listed_in", "description",
    "content_language", "_batch_id", "_ingestion_sequence", "_source_system", "_source_file", "_ingested_at"
]

for column_name in known_columns:
    if column_name not in bronze_df.columns:
        if column_name == "_ingestion_sequence":
            bronze_df = bronze_df.withColumn(column_name, F.lit(None).cast("long"))
        else:
            bronze_df = bronze_df.withColumn(column_name, F.lit(None).cast("string"))

silver_updates = (
    bronze_df
    .select(*known_columns)
    .withColumn("show_id", clean_string("show_id"))
    .withColumn("content_type", clean_string("type"))
    .withColumn("title", clean_string("title"))
    .withColumn("director", clean_string("director"))
    .withColumn("cast", clean_string("cast"))
    .withColumn("country", clean_string("country"))
    .withColumn("rating", clean_string("rating"))
    .withColumn("genres", clean_string("listed_in"))
    .withColumn("description", clean_string("description"))
    .withColumn("content_language", clean_string("content_language"))
    .withColumn("date_added", F.to_date(F.trim(F.col("date_added")), "MMMM d, yyyy"))
    .withColumn("release_year", F.expr("try_cast(release_year AS INT)"))
    .withColumn("duration_raw", clean_string("duration"))
    .withColumn("duration_digits", F.regexp_extract(F.col("duration"), r"^(\d+)", 1))
    .withColumn("duration_value", F.expr("try_cast(duration_digits AS INT)"))
    .withColumn("duration_unit", F.regexp_extract(F.col("duration"), r"\d+\s+(.*)$", 1))
    .withColumn("silver_updated_at", F.current_timestamp())
    .drop("type", "listed_in", "duration", "duration_digits")
)

business_columns = [
    "show_id", "content_type", "title", "director", "cast", "country",
    "date_added", "release_year", "rating", "duration_raw", "duration_value",
    "duration_unit", "genres", "description", "content_language"
]

silver_updates = silver_updates.withColumn(
    "_record_hash",
    F.sha2(F.concat_ws("||", *[F.coalesce(F.col(c).cast("string"), F.lit("")) for c in business_columns]), 256)
)


In [0]:
show_id_valid = F.col("show_id").isNotNull()
title_valid = F.col("title").isNotNull()
content_type_valid = F.coalesce(F.col("content_type").isin("Movie", "TV Show"), F.lit(False))
release_year_valid = F.coalesce(F.col("release_year").between(1900, F.year(F.current_date()) + 1), F.lit(False))

raw_quality_filter = show_id_valid & title_valid & content_type_valid & release_year_valid
quality_filter = F.coalesce(raw_quality_filter, F.lit(False))

valid_updates = silver_updates.filter(quality_filter)
invalid_updates = (
    silver_updates
    .filter(~quality_filter)
    .withColumn(
        "_quality_errors",
        F.concat_ws(
            "; ",
            F.when(~show_id_valid, F.lit("show_id is null or empty")),
            F.when(~title_valid, F.lit("title is null or empty")),
            F.when(~content_type_valid, F.lit("content_type is not Movie or TV Show")),
            F.when(~release_year_valid, F.lit("release_year is missing or outside accepted range")),
        )
    )
    .withColumn("_quarantined_at", F.current_timestamp())
)

dedup_window = (
    Window
    .partitionBy("show_id")
    .orderBy(
        F.col("_ingested_at").desc_nulls_last(),
        F.col("_ingestion_sequence").desc_nulls_last(),
        F.col("_batch_id").desc_nulls_last()
    )
)

deduplicated_updates = (
    valid_updates
    .withColumn("_row_number", F.row_number().over(dedup_window))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
    .withColumn("_prepared_at", F.current_timestamp())
)

print("Valid updates after dedup:", deduplicated_updates.count())
print("Rejected rows in this run:", invalid_updates.count())


In [0]:
# Persist rejected rows for audit and debugging.
(
    invalid_updates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_quarantine_table)
)

display(spark.table(silver_quarantine_table).select("show_id", "title", "content_type", "release_year", "_quality_errors"))

(
    deduplicated_updates.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_updates_table)
)

prepared_updates = spark.table(silver_updates_table)
display(prepared_updates.orderBy("show_id").limit(20))
